# SkillRadar – Del 2: Data, O*NET og preprocessing
## Teknisk gennemgang til mundtlig eksamen

**Formål:** At kunne forklare præcis, hvor data kommer fra, hvordan de er undersøgt, hvordan de behandles, og hvorfor appen kan beregne efterspørgsel efter navngivne teknologier.

**Grundlag:** SkillRadar-repositoriets `README.md`, `skillradar/demand.py`, `skillradar/matcher.py`, `main.py` samt jeres to oprindelige notebooks `data_inspection_primary.ipynb` og `data_inspection_secondary.ipynb`. Koden er gennemgået på GitHub den 5. november 2026.

**Sådan læser du notebooken**
- **Fra projektet:** beskrivelse eller uddrag af jeres faktiske implementering.
- **Kørbart eksempel:** lille, syntetisk datasæt der demonstrerer samme princip uden store Kaggle-filer.
- **Eksamensforklaring:** noget, du kan forklare med dine egne ord.
- **Alternativer / begrænsninger:** afvejninger, som er relevante til kritiske spørgsmål.

Kør cellerne oppefra og ned. De kørbare eksempler bruger kun Python samt `pandas` og `numpy`, som jeres projekt allerede anvender. De ændrer **ikke** jeres projektdata og kræver **ikke** downloads af Kaggle-datasættene.

## 1. Det store overblik: To forskellige pipelines

Det er vigtigt at skelne mellem **offline dataforberedelse** og **online brug af appen**.

**Offline, når data klargøres:**

1. Rå LinkedIn-postings downloades fra Kaggle.
2. O*NET giver et kontrolleret ordforråd af teknologier.
3. Jobtitler bruges til at finde fem relevante roller.
4. Beskrivelser af jobopslag læses, og samme `KeywordMatcher` søger efter teknologier.
5. For hvert opslag gemmes metadata, et tekstfingeraftryk og de fundne teknologier.
6. De små tabeller gemmes som `posting_tools_primary.csv` og `posting_tools_secondary.csv`.

**Online, når en bruger åbner appen:**

1. Appen læser de **allerede behandlede** posting-tabeller.
2. Den filtrerer til den valgte rolle og eventuelt entry-level.
3. Den udregner, hvor stor en andel af opslagene der nævner hver teknologi.
4. De andele sammenlignes med de teknologier, brugeren bekræfter fra sit CV.

**Hvorfor skellet?** Et datagrundlag på over en million opslag og en stor tekstfil bør ikke genbehandles ved hvert klik i en webapp. Denne arkitektur gør appen langt hurtigere.

## 2. Hvilke filer gør hvad?

| Projektfil | Rolle i Del 2 |
|---|---|
| `preprocessing/data_inspection_primary.ipynb` | Undersøger det store datasæt, skill labels, timestamps, tekst og matcherens resultater |
| `preprocessing/data_inspection_secondary.ipynb` | Undersøger det andet datasæt, metadata, O*NET og den første baseline |
| `skillradar/matcher.py` | Definerer ordforråd, aliaser og extraction af teknologier fra tekst |
| `skillradar/demand.py` | Bygger posting→tools-tabeller, renser/fjerner dubletter, filtrerer og udregner demand |
| `main.py` | Kommandolinje, herunder `build --source primary/secondary` |
| `data/onet/software_skills.csv` | O*NET-teknologier, som matcheren tager udgangspunkt i |
| `data/processed/posting_tools_primary.csv` | Klargjort, lille tabel fra primære opslag |
| `data/processed/posting_tools_secondary.csv` | Klargjort, lille tabel fra sekundære opslag |
| `app.py` | Bruger de færdige tabeller; bygger dem ikke ved CV-upload |

**Eksamen:** En data-inspection notebook er undersøgende arbejde. Den er ikke nødvendigvis identisk med den senere produktionspipeline i `demand.py`. Beskriv derfor begge – og fortæl, hvordan undersøgelsen førte til beslutninger i implementeringen.

## 3. Hvor kommer datasættene fra?

### Primary: asaniczka / 1.3M LinkedIn Jobs & Skills 2024
- Omtrent **1,3 millioner** indsamlede LinkedIn-opslag i hele datasættet.
- Et **snapshot fra januar 2024**, ikke 12 måneders løbende observationer.
- Indeholder opslag fra bl.a. USA, Storbritannien, Canada og Australien; ca. 85 % af de rå opslag er fra USA ifølge projektets README.
- Relevante råfiler:
  - `linkedin_job_postings.csv` med titel, `job_link`, lokation/land, niveau og metadata.
  - `job_summary.csv` med opslagsteksten; flere GB.
  - `job_skills.csv` med datasetleverandørens egne, støjende skill-lister.

### Secondary: arshkon / LinkedIn Job Postings
- Omtrent **124.000** opslag i det rå datasæt.
- Primært et **snapshot fra april 2024**, USA.
- `postings.csv` indeholder titler, beskrivelser og erfaringsniveau.
- Har `Entry level` og `Internship`, som primary ikke giver samme mulighed for at udvælge.
- Dets `jobs/job_skills.csv` og `mappings/skills.csv` beskriver kun brede skill-kategorier, **ikke** de konkrete teknologier, SkillRadar vil rangere.

**Hvorfor to?** Primary giver større stikprøver for flere målroller; secondary giver et uafhængigt datasæt til robusthed og mulighed for at se juniorstillinger.

**Vigtigt:** Tallet på 1,3 mio. er størrelsen af *hele* kildedatasættet – ikke antallet af opslag, der indgår i én bestemt rolle i rapporten.

### Kildekritik og repræsentativitet

**Fra jeres README:** Disse er scrape-baserede udtræk, ikke et tilfældigt udvalg af hele arbejdsmarkedet. Data er fra 2024 og overvejende amerikanske. Derfor er det fagligt upræcist at sige, at resultaterne viser den **nuværende** danske efterspørgsel.

Det appen faktisk kan sige: *»I det valgte udsnit af LinkedIn-opslag fra 2024 nævnes SQL i X % af opslagene, som vores rollematcher klassificerer som data scientist.«*

Det appen **ikke** kan konkludere: At X % af alle danske arbejdsgivere kræver SQL i dag, eller at en person med 70 % coverage har 70 % sandsynlighed for at blive ansat.

**Alternativer:** Danske jobopslag, ESCO som europæisk taxonomi, nyere løbende indsamlede data eller stratificeret sampling. De ville forbedre relevansen, men øge indsamling, licensarbejde og databehandling.

## 4. Hvorfor begyndte I med data inspection?

Begge notebooks tjekker tidligt de risici, projektet er afhængigt af:
- Er der faktisk tekst til at udtrække teknologier fra?
- Hvor mange opslag findes der for hver målrolle?
- Er der missing values, dubletter, mærkelige datafelter eller problemer med formater?
- Er tidsdækningen lang nok til at måle trends?
- Har O*NET ord, der kan genkendes i virkelig CV-/jobtekst?

I anvender bl.a. `pd.read_csv`, `shape`, datatyper, manglende værdier, antal unikke værdier og `value_counts`. Disse er **exploratory data analysis (EDA)** og kontrollerer, om det er forsvarligt at bygge resten af modellen på kilderne.

I primary-notebooken undersøger I også, hvor forskelligt jobopslagets fritekst og Kaggle-kildens egne skill labels opfører sig; den analyse bidrager til, at projektet ender med én fælles matcher.

## 5. Data-inspection i et lille eksempel
**Kørbar illustration, ikke jeres rigtige jobdata.**

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

raw_jobs = pd.DataFrame([
    {"job_id": 1, "title": "Data Scientist", "description": "Python SQL scikit-learn", "level": "Entry level"},
    {"job_id": 2, "title": "Data Scientist", "description": "Python SQL", "level": "Mid-Senior level"},
    {"job_id": 3, "title": "Software Engineer", "description": None, "level": "Entry level"},
    {"job_id": 3, "title": "Software Engineer", "description": None, "level": "Entry level"},
    {"job_id": 4, "title": None, "description": "SQL", "level": None},
])
print("Rækker og kolonner:", raw_jobs.shape)
print("\nManglende værdier:")
print(raw_jobs.isna().sum())
print("\nDublerede job_id:", raw_jobs["job_id"].duplicated().sum())
display(raw_jobs)

Rækker og kolonner: (5, 4)

Manglende værdier:
job_id         0
title          1
description    2
level          1
dtype: int64

Dublerede job_id: 1


,job_id,title,description,level
0,1,Data Scientist,Python SQL scikit-learn,Entry level
1,2,Data Scientist,Python SQL,Mid-Senior level
2,3,Software Engineer,None,Entry level
3,3,Software Engineer,None,Entry level
4,4,None,SQL,None


### Forklaring af output

- `shape` viser `(5, 4)`: fem rækker og fire kolonner.
- `isna().sum()` viser hvor meget information der mangler, **pr. kolonne**. Det er vigtigt, fordi matching ikke kan bruge en manglende beskrivelse.
- `duplicated()` afslører, at to rækker har samme `job_id`. Det er ikke det samme som at afsløre identiske annoncer med forskellige ID'er – det kommer senere.
- Tabellen gør det synligt, at et opslag godt kan have titel men ingen tekst.

**Hvorfor denne metode?** EDA giver en hurtig kontrol, inden der skrives mere kompleks preprocessing.  
**Alternativer:** `df.info()`, `describe(include="all")`, automatiske profiler som `ydata-profiling`, validering med Pandera. Sidstnævnte er bedre til faste produktionskrav, men tungere end den første undersøgelse.

## 6. Tidsdækning: hvorfor I droppede »trends over tid«

I de tidlige projektidéer overvejede I trends. I begge inspection notebooks undersøges tidsfelter:
- Primary: `first_seen` (hvornår scraberen først observerede opslaget).
- Secondary: `listed_time` og `original_listed_time` (Unix-timestamps i **millisekunder**).

En enkelt scrape kan indeholde varierende offentliggørelsesdatoer, men det gør **ikke** automatisk data til en repræsentativ månedlig tidsserie.

I sammenlignede januar- og april-snapshots og lavede en foreløbig test af forskelle. I README ender I eksplicit med **ingen »trending up/down«-påstande**: forskelle mellem scraperne, landefordeling, rolleudvalg og opslagssammensætning kan let forveksles med markedsudvikling.

Dette er et stærkt metodisk eksamenspunkt: I ændrede projektets scope på baggrund af konkrete datakvalitetsresultater.

In [2]:
# Kørbart mini-eksempel: millisekunder vs. ISO-datoer
timestamps_ms = pd.Series([1712340000000, 1712600000000])
print("Korrekt med unit='ms':")
print(pd.to_datetime(timestamps_ms, unit="ms", utc=True).dt.date.to_list())
print("\nForkert fortolkning som nanosekunder:")
print(pd.to_datetime(timestamps_ms, unit="ns", utc=True).dt.date.to_list())

Korrekt med unit='ms':
[datetime.date(2024, 4, 5), datetime.date(2024, 4, 8)]

Forkert fortolkning som nanosekunder:
[datetime.date(1970, 1, 1), datetime.date(1970, 1, 1)]


**Forklaring:** Pandas kan ikke altid gætte tidsenheden korrekt. Med `unit="ms"` konverteres sekundærkildens millisekundtal til forventede datoer i 2024. Hvis tallene i stedet læses som nanosekunder, havner man tæt på 1970. Forkerte tidsstempler kan ødelægge al tidsanalyse.

**Alternativ:** kontrollér min/max mod kildens dokumentation og indfør assertions, fx at årstallene ligger i et forventet interval.

## 7. O*NET: Hvad er det, og hvad bruger I?

**O*NET** er en amerikansk klassifikation af erhverv, arbejdskrav og software/teknologier. I projektet er den centrale fil `software_skills.csv`, hvor teknologier (fx `Workplace Example`) knyttes til erhverv og kan være markeret `Hot Technology` eller `In Demand`.

Andre O*NET-filer blev undersøgt i secondary-notebooken:
- `occupation_data.csv`: erhvervsbetegnelser og SOC-koder.
- `job_titles.csv`: alternative titler for erhverv.
- `essential_skills.csv` og `transferable_skills.csv`: mere generelle færdigheder, men de bruges **ikke** som hovedvokabular for appens værktøjsrapport.

**Vigtigt skel:**
- O*NET bestemmer **hvilke kanoniske teknologinavne I kan genkende**.
- Jobopslagene bestemmer **hvor ofte disse teknologier faktisk nævnes i det valgte udsnit**.
- En O*NET-etiket som »Hot Technology« betyder **ikke**, at jeres beregnede demand automatisk er høj i den valgte rolle.

## 8. Hvad er normalisering og aliaser?

O*NET bruger ofte lange navne, som er gode til en formel database, men ikke matcher normalt sprog uden justeringer. I koden beskriver I bl.a.:

- `Microsoft Power BI` → også alias `Power BI`.
- `Structured query language SQL` → også alias `SQL`, når forkortelsen kan begrundes.
- `Amazon Web Services AWS software` → fx forkortede former, når alias-reglerne tillader det.

**Kanonisk navn:** fælles identitet i rapporten.  
**Alias:** alternativ skrivemåde, der skal føre til samme identitet.

**Hvorfor?** I et CV kan der stå `SQL`, mens O*NET bruger et længere navn. Uden aliaser ville samme teknologi fejlagtigt fremstå som fraværende.

**Begrænsning:** Reglerne kan også skabe falske positive. `Go` kan være programmeringssprog eller et almindeligt engelsk ord. `Google` kan være et virksomhedsnavn frem for en navngiven teknologi.

In [3]:
# Kørbart pædagogisk eksempel på kanoniske teknologier – IKKE en kopi af hele matcher.py
canonical = {
    "Structured query language SQL": {"sql", "structured query language sql"},
    "Microsoft Power BI": {"power bi", "microsoft power bi"},
    "Amazon Web Services AWS software": {"aws", "amazon web services"},
}

examples = ["I build dashboards in Power BI.", "SQL and AWS experience"]
for text in examples:
    simple_found = [tool for tool, names in canonical.items()
                    if any(alias in text.lower() for alias in names)]
    print(text, "→", simple_found)

I build dashboards in Power BI. → ['Microsoft Power BI']
SQL and AWS experience → ['Structured query language SQL', 'Amazon Web Services AWS software']


### Hvad dette eksempel viser – og ikke viser

Det viser princippet bag kanoniske navne og aliaser: flere skrivemåder kan pege på ét værktøj.

Det er **bevidst simplere** end jeres rigtige `KeywordMatcher`. Her bruges naiv substring-søgning, som fx ville kunne finde `R` inde i andre ord. I den faktiske kode bruger I tokenisering, n-grams, case-regler, manuel alias-fjernelse og konfliktløsning. Derfor må man **ikke** bruge dette eksempel som dokumentation for den faktiske matchers præcision.

## 9. Sådan bygger jeres matcher sit ordforråd – faktisk kode

Uddrag fra `skillradar/matcher.py` (hovedidéen):

```python
flags = software.groupby("Workplace Example")[["Hot Technology", "In Demand"]].agg(
    lambda s: (s == "Y").any()
)
important = flags.index[flags["Hot Technology"] | flags["In Demand"]]
vocab = sorted(t for t in important if not is_generic(t))
```

**Linje for linje:**
1. `groupby("Workplace Example")` samler alle rækker om samme navngivne teknologi. Den samme teknologi kan forekomme for mange occupations.
2. `(s == "Y").any()` betyder: er teknologien markeret med `Y` i **mindst én** tilknytning?
3. `|` er logisk OR: enten `Hot Technology` **eller** `In Demand` er tilstrækkeligt.
4. `is_generic(t)` sorterer generiske kategorier fra. Formålet er at sammenligne **navngivne værktøjer/teknologier**, ikke brede kompetencer som »teamwork«.
5. `sorted(...)` giver et konsistent ordforråd.

**Hvorfor OR frem for AND?** AND ville være mere restriktiv og kunne udelukke relevante værktøjer. OR er mere dækkende, men kan også give et bredere katalog af kandidater.

**Muligt alternativ:** Tag alle O*NET-værktøjer og lad empirisk efterspørgsel filtrere bagefter. Det øger coverage, men også risikoen for generiske eller sjældne match.

In [4]:
software_demo = pd.DataFrame([
    {"Workplace Example": "Python", "Hot Technology": "Y", "In Demand": "N"},
    {"Workplace Example": "Python", "Hot Technology": "N", "In Demand": "Y"},
    {"Workplace Example": "SQL", "Hot Technology": "N", "In Demand": "Y"},
    {"Workplace Example": "Old Software", "Hot Technology": "N", "In Demand": "N"},
])
flags_demo = software_demo.groupby("Workplace Example")[["Hot Technology", "In Demand"]].agg(
    lambda s: (s == "Y").any()
)
display(flags_demo)
print("Udvalgte navne:", flags_demo.index[
    flags_demo["Hot Technology"] | flags_demo["In Demand"]
].tolist())

,Hot Technology,In Demand
Workplace Example,,
Old Software,False,False
Python,True,True
SQL,False,True


Udvalgte navne: ['Python', 'SQL']


**Output:** `Python` er markeret, fordi mindst én af dens O*NET-rækker har et `Y`. `SQL` kommer også med, mens `Old Software` ikke gør. I den virkelige funktion anvendes *derudover* `is_generic`, så resultatet kan blive yderligere indsnævret.

## 10. Hvad gør I ved fejlagtige ordmatches?

I `matcher.py` er der konkrete sikkerhedsregler:
- `CASE_SENSITIVE` håndterer ord, hvor store/små bogstaver betyder noget, fx `R`, `Go`, `Excel`.
- `MANUAL_DROP = {"Google", "Facebook"}` fjerner aliaser, der ifølge jeres manuelle opslagstest hyppigt ramte virksomhedsnavne/sociale medier.
- `TOKEN` sikrer, at tokens som `C++`, `C#` og `Node.js` kan håndteres mere meningsfuldt.
- `MAX_NGRAM = 8` begrænser længden af sammenhængende tokens, der prøves.

**Fra jeres README:** I undersøgte eksempeltekster for tvetydige aliaser og fjernede nogle manuelt. I droppede ikke automatisk alle ord, der optrådte meget ofte, fordi det fx kunne komme til at fjerne en reel teknologi som Excel.

**Alternativer:** Word boundaries alene, regelbaseret NLP med kontekst, supervised named-entity recognition eller LLM-/hybridmatching. Den manuelle regelbase er hurtig, forklarbar og uden API-kald, men kræver vedligehold og overser teknologiord uden for ordforrådet.

## 11. Rollefiltrering: Kun relevante opslag må tælles

I `skillradar/demand.py` har I en ordbog med fem målroller:

```python
ROLE_PATTERNS = {
    "data scientist":    r"\bdata scien",
    "data analyst":      r"\bdata analy",
    "software engineer": r"\bsoftware (?:engineer|developer)",
    "ml engineer":       r"\b(?:machine learning|ml) engineer",
    "ux designer":       r"\b(?:ux|ui|uxui|user experience)\b.*design",
}
```

Funktionen `tag_role()` går gennem disse patterns i rækkefølge og tildeler hver titel den **første** matchende rolle. Titler, der ikke matcher, får `NaN` og sorteres fra ved opbygningen.

**Hvorfor regular expressions?** Titler har variationer: »Data Scientist«, »Senior Data Scientist« og »Software Developer«. Regex kan håndtere dette uden at vedligeholde en komplet liste af præcise titler.

**Faldgrube:** `\b` (word boundary) blev vigtigt efter en fejl med UX/UI, hvor `ui` i ord som »Building« kunne give fejlagtige matches. Ifølge jeres README gav den tidligere regel mange irrelevante UX-hits.

**Alternativer:** manuel titelliste, O*NET occupation-linking, embeddings eller en klassifikationsmodel. Regex er transparent og let at teste, men kan misse uventede titler og overlappende roller.

In [5]:
import re
patterns = {
    "data scientist": r"\bdata scien",
    "data analyst": r"\bdata analy",
    "software engineer": r"\bsoftware (?:engineer|developer)",
    "ml engineer": r"\b(?:machine learning|ml) engineer",
    "ux designer": r"\b(?:ux|ui|uxui|user experience)\b.*design",
}
titles = pd.Series([
    "Senior Data Scientist", "Software Developer", "ML Engineer",
    "UX Designer", "Building Designer", "Data Analyst", "Accountant"
])
roles = pd.Series([np.nan] * len(titles), dtype="object")
for name, pattern in patterns.items():
    hit = roles.isna() & titles.str.contains(pattern, case=False, regex=True, na=False)
    roles.loc[hit] = name
display(pd.DataFrame({"title": titles, "role": roles}))

,title,role
0,Senior Data Scientist,data scientist
1,Software Developer,software engineer
2,ML Engineer,ml engineer
3,UX Designer,ux designer
4,Building Designer,NaN
5,Data Analyst,data analyst
6,Accountant,NaN


**Output:** Fem titler klassificeres som forventet, mens `Building Designer` og `Accountant` ikke falder i jeres fem målroller. Det er en vigtig forskel mellem »ingen værktøjer i teksten« og »opslaget er slet ikke med i analysen«.

**Eksamen:** En forkert rolletildeling påvirker både tælleren (hvor mange opslag med et værktøj) og nævneren (samlet antal opslag i rollen).

## 12. Experience levels: Hvorfor normaliseres de?

Datasættene bruger forskellige labels:
- Primary: `Mid senior`, `Associate`.
- Secondary: `Entry level`, `Internship`, `Mid-Senior level`, `Director`, `Executive`.

I `LEVEL_MAP` bliver labels mappet til fælles interne værdier, bl.a. `mid-senior`, `entry`, `internship`. `normalise_level()` fjerner overflødige mellemrum og bruger lowercase, før den slår op.

**Hvorfor?** Ellers ville et filter efter `entry` ikke kunne finde kildens `Entry level`.

**Alternativer:** behold originale labels og oversæt dem i hver query; eller opret en egentlig dimensionstabel. Det sidste er mere skalerbart, men unødigt for de få kategorier her.

In [6]:
LEVEL_MAP_DEMO = {
    "mid senior": "mid-senior", "associate": "associate",
    "entry level": "entry", "internship": "internship",
    "mid-senior level": "mid-senior", "director": "director",
    "executive": "executive",
}
levels = pd.Series([" Entry level ", "Internship", "Mid senior", "Mid-Senior level", None])
normalised = levels.str.strip().str.lower().map(
    lambda x: LEVEL_MAP_DEMO.get(x, x) if isinstance(x, str) else x
)
display(pd.DataFrame({"original": levels, "normaliseret": normalised}))

,original,normaliseret
0,Entry level,entry
1,Internship,internship
2,Mid senior,mid-senior
3,Mid-Senior level,mid-senior
4,None,None


## 13. Offline build: Primary-filen

Dette er rækkefølgen i jeres **faktiske** `build_primary()` i `skillradar/demand.py`:

1. Læs metadata fra `linkedin_job_postings.csv`, bl.a. `job_link`, `job_title`, `job_level`, `search_country` og eventuelt `company`.
2. `drop_duplicates("job_link")`: behold ét metadataopslag pr. link.
3. `tag_role(meta["job_title"])`: klassificér titler.
4. Behold kun opslag med en af målrollerne.
5. Brug `job_link` som nøgle (`set_index("job_link")`).
6. Læs `job_summary.csv` i `chunksize=100_000`.
7. Behold kun tekst for relevante `job_link` og fjern dubletter inden for chunk.
8. For hver relevant tekst køres `matcher.extract_tools(text)`.
9. Skriv én række med posting-id, rolle, niveau, land, titel, eventuelt firma, `text_hash` og semikolonadskilte værktøjer.
10. Normalisér erfaringstrin og returnér tabellen.

**Hvorfor chunks?** `job_summary.csv` er en meget stor fil (flere GB). Ved chunking ligger kun en afgrænset del i hukommelsen ad gangen.

**Nuance:** `build_primary()` deduplikerer metadata på `job_link`, men den efterfølgende tekstbaserede deduplikering foregår i `load_posting_tools()` – ikke som fuld tekst-deduplikering i `build_primary()` selv.

In [7]:
# Kørbar illustration af chunking – uden at læse fra disk
chunks = [
    pd.DataFrame({"job_link": ["a", "b"], "text": ["Python SQL", "Java"]}),
    pd.DataFrame({"job_link": ["c", "d"], "text": ["Python", "SQL Docker"]}),
]
relevant_links = {"a", "c"}
selected = []
for chunk in chunks:
    selected.extend(chunk[chunk["job_link"].isin(relevant_links)].to_dict("records"))
display(pd.DataFrame(selected))

,job_link,text
0,a,Python SQL
1,c,Python


**Output:** Kun `a` og `c` bliver behandlet. Pointen er, at man ikke behøver indlæse hele tekstmængden for at udvælge relevante job.

**Alternativer:** Polars lazy API, DuckDB `read_csv_auto`, Dask, Parquet med partitionering eller database. De kan være mere effektive eller skalerbare, men kræver mere infrastruktur.

## 14. Offline build: Secondary-filen

Den faktiske `build_secondary()` adskiller sig fra `build_primary()`:

1. Læser én samlet `postings.csv` med `job_id`, `title`, `description`, `skills_desc` og `formatted_experience_level`.
2. Fjerner dublerede `job_id`.
3. Tildeler en målrolle via titel.
4. Sorterer titler uden målrolle fra.
5. Samler **`description` + newline + `skills_desc`**, hvor missing tekst først bliver tom streng.
6. Bruger matcher på den sammenlagte tekst.
7. Gemmer `posting_id`, `source="secondary"`, rolle, normaliseret niveau, `country="United States"`, titel, firma, hash og værktøjer.

**Hvorfor forskellig kode for kilderne?** Datasættenes skemaer er faktisk forskellige. Primary gemmer beskrivelsen i en separat kæmpestor fil, mens secondary har beskrivelsen i hovedtabellen.

**Alternativ:** Indfør ensartede adapterklasser med samme outputskema. Det er især interessant, hvis en tredje datakilde senere skal kunne tilføjes.

## 15. Hvad er et tekstfingeraftryk, og hvorfor fjerner I dubletter?

Fra jeres `text_hash()`:

```python
def text_hash(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    return hashlib.sha1(" ".join(text.lower().split()).encode("utf-8")).hexdigest()[:16]
```

**Præcis forklaring:**
- `text.lower()` gør teksten case-insensitiv for fingerprintet.
- `.split()` og `" ".join(...)` normaliserer almindelige whitespace-forskelle.
- `.encode("utf-8")` gør strengen til bytes.
- SHA-1 beregner en deterministisk hash.
- `[:16]` beholder kun de første 16 hex-tegn.
- Tom/manglende tekst får `""`, som behandles særskilt ved deduplikering.

**Formålet:** Den samme annonce kan være genopslået i flere byer med forskellige ID'er. Hvis den tælles flere gange, kan enkelte arbejdsgivere få for stor vægt i demand-statistikken.

**Nuance:** Det er en hash til praktisk dubletgenkendelse, ikke en garanti for identisk semantik eller en datasikkerhedsmekanisme. Små tekstændringer kan producere forskellige hashes.

In [8]:
import hashlib

def text_hash_demo(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    return hashlib.sha1(" ".join(text.lower().split()).encode("utf-8")).hexdigest()[:16]

texts = [
    "We need Python and SQL",
    "we   need PYTHON and SQL",
    "We need Python and Docker",
]
for t in texts:
    print(text_hash_demo(t), " | ", t)
print("Første to ens?", text_hash_demo(texts[0]) == text_hash_demo(texts[1]))

65708ec59cbad3c5  |  We need Python and SQL
65708ec59cbad3c5  |  we   need PYTHON and SQL
1f29ecfd8f005fb3  |  We need Python and Docker
Første to ens? True


**Output:** Første og anden tekst får samme fingerprint, fordi deres forskelle kun handler om case og whitespace. Den tredje tekst får en anden hash.

**Alternativer:** Exact hash af rå tekst (mere følsom), MinHash/LSH til næsten-identiske tekster, fuzzy matching med Levenshtein-lignende afstande eller kombinerede nøgler (`company + title + location`). Near-duplicate-metoder opdager flere genopslag, men kan også slå forskellige jobs sammen.

## 16. Hvornår fjernes de tekstbaserede dubletter?

I den faktiske `load_posting_tools()` sker bl.a.:

```python
dup = df.duplicated(["role", "text_hash"]) & (df["text_hash"] != "")
df = df[~dup].reset_index(drop=True)
```

**To centrale detaljer:**
- Hashen sammenlignes **inden for den samme rolle**. Identisk tekst i to forskellige roller slettes ikke automatisk som én og samme observation.
- Tomme hashes deduplikeres ikke ved den regel.

**Fra projektets README:** Den senere dubletkontrol fjernede 4.049 opslag i primary og 342 i secondary blandt de relevante rolleopslag. Tallene er projektets dokumenterede analyseopgørelse, ikke beregnet på det syntetiske eksempel her.

**Metodisk svaghed:** Deduplikering kan ændre både numerator og denominator. Hvis mange dubletter nævner Python, vil Python-andelen typisk falde, når de fjernes. Derfor er deduplikering ikke blot kosmetisk oprydning.

In [9]:
demo_posts = pd.DataFrame([
    {"role": "data scientist", "posting_id": "a", "text": "Python SQL"},
    {"role": "data scientist", "posting_id": "b", "text": " python   sql "},
    {"role": "data scientist", "posting_id": "c", "text": "Python Docker"},
    {"role": "software engineer", "posting_id": "d", "text": "Python SQL"},
])
demo_posts["text_hash"] = demo_posts["text"].map(text_hash_demo)
is_dup = demo_posts.duplicated(["role", "text_hash"]) & demo_posts["text_hash"].ne("")
display(demo_posts.assign(removed=is_dup))
display(demo_posts.loc[~is_dup].reset_index(drop=True))

,role,posting_id,text,text_hash,removed
0,data scientist,a,Python SQL,f8ea0b694ec845f2,False
1,data scientist,b,python sql,f8ea0b694ec845f2,True
2,data scientist,c,Python Docker,a92cc13e8969bfce,False
3,software engineer,d,Python SQL,f8ea0b694ec845f2,False


,role,posting_id,text,text_hash
0,data scientist,a,Python SQL,f8ea0b694ec845f2
1,data scientist,c,Python Docker,a92cc13e8969bfce
2,software engineer,d,Python SQL,f8ea0b694ec845f2


## 17. Output fra build: én række pr. relevant posting

Når `main.py build --source primary` eller `--source secondary` køres, kaldes `build()` i `demand.py`. Den vælger passende build-funktion og skriver:

```
data/processed/posting_tools_primary.csv
data/processed/posting_tools_secondary.csv
```

Hver række indeholder felterne:

| Felt | Betydning |
|---|---|
| `posting_id` | Identifikator fra kilden |
| `source` | `primary` eller `secondary` |
| `role` | Standardiseret rolle |
| `level` | Standardiseret erfaringsniveau |
| `country` | Land (secondary er sat til United States) |
| `title` | Jobtitel |
| `company` | Firma, hvis tilgængeligt |
| `text_hash` | Hash af posting-teksten |
| `tools` | Fundne kanoniske teknologinavne, sorteret og samlet med `;` |

**Hvorfor er `tools` én CSV-streng?** Det er en enkel serialisering af et sæt fundne teknologier i en kompakt tabel. Ved indlæsning splittes den tilbage til et Python-`set`.

**Alternativ:** Relationel »posting_tools«-koblingstabel med én række pr. (`posting_id`, `tool`) eller Parquet med list-kolonner. Det er mere normaliseret, men kræver mere datalogik.

In [10]:
row = {
    "posting_id": "job-001",
    "role": "data scientist",
    "tools": ";".join(sorted({"Python", "SQL", "Docker"})),
}
display(pd.DataFrame([row]))
restored = set(row["tools"].split(";"))
print("Gendannet set:", restored)

,posting_id,role,tools
0,job-001,data scientist,Docker;Python;SQL


Gendannet set: {'SQL', 'Docker', 'Python'}


## 18. Efter indlæsning: Hvorfor et `set`, ikke en `list`?

Når `load_posting_tools()` læser CSV'en, laver den:

```python
df["tool_set"] = df["tools"].map(
    lambda s: set(s.split(SEP)) if s else set()
)
```

Hvis et opslag skriver »Python« syv gange, må opslaget stadig kun bidrage med **én** Python-observation.

Et `set` har unikke elementer, hvilket passer præcis til jeres definition af efterspørgsel: *andelen af opslag, der nævner værktøjet mindst én gang*.

**Alternativ metric:** Samlet antal omtaler af et værktøj. Den ville belønne lange jobopslag og gentagne buzzwords og ville ikke længere kunne fortolkes som »X % af jobopslagene«.

## 19. Demand: Hvad betyder procenttallet faktisk?

Jeres mål er en andel af **opslag**, ikke en andel af **alle ord**, en andel af **arbejdsgivere** eller en sandsynlighed for at blive ansat.

For rolle `r` og teknologi `t`:

\[
\widehat p(t\mid r) = \frac{\#\{\text{opslag i rolle }r\text{ med }t\}}{\#\{\text{opslag i rolle }r\}}
\]

I `demand.py` sker dette i `demand_from_sets()`:
- Funktionen modtager en liste af sets, ét set pr. posting.
- `n = len(tool_sets)` er antal opslag (nævneren).
- En `Counter`-lignende optælling tæller hver teknologi én gang pr. set.
- `share = k / n` er efterspørgselsandelen.
- Resultatet sorteres faldende efter `share`.

**Eksamen:** Hvis SQL står i 40 af 100 opslag, er SQL-demand 40 %. Det siger intet om, hvor mange gange ordet optræder i hver annonce.

In [11]:
toy_postings = [
    {"Python", "SQL"},
    {"Python", "SQL"},
    {"Python", "Docker"},
    {"SQL"},
    {"Python", "Python"},  # et set gør dubletten til én observation
]
n = len(toy_postings)
counts = Counter(tool for tools in toy_postings for tool in tools)
demand_demo = pd.DataFrame([
    {"tool": tool, "postings_with_tool": count, "total_postings": n, "share": count/n}
    for tool, count in counts.items()
]).sort_values(["share", "tool"], ascending=[False, True]).reset_index(drop=True)
display(demand_demo)

,tool,postings_with_tool,total_postings,share
0,Python,4,5,0.8
1,SQL,3,5,0.6
2,Docker,1,5,0.2


**Output:** Python optræder i 4/5 opslag (80 %), SQL i 3/5 (60 %) og Docker i 1/5 (20 %). Det er beregnet efter samme *princip* som `demand_from_sets()`.

**Begrænsning:** Man må ikke lægge de tre andele sammen og forvente 100 %. Ét jobopslag kan nævne flere teknologier, så andelene overlapper.

## 20. Wilson 95 %-intervaller: Hvorfor viser I usikkerhed?

Jeres kode anvender Wilson score-intervallet i funktionen `wilson(k, n, z=1.96)`, hvor `k` er antal opslag med værktøjet og `n` er antal relevante opslag.

Metoden giver et interval omkring stikprøveandelen. Det fungerer generelt bedre end det naive normal-approksimationsinterval, særligt ved små n eller andele tæt på 0 og 1.

Formlen i koden er:

\[
\text{centre} = \frac{\widehat p+z^2/(2n)}{1+z^2/n}
\]

\[
\text{half} = \frac{z\sqrt{\widehat p(1-\widehat p)/n+z^2/(4n^2)}}{1+z^2/n}
\]

\[
[\max(0,\text{centre}-\text{half}),\min(1,\text{centre}+\text{half})]
\]

**Vigtigt:** Et 95 %-konfidensinterval har en *frekventistisk* fortolkning under modellens antagelser. Det er ikke en garanti om, at det sande tal er i intervallet med 95 % sandsynlighed for netop dette udtræk. Og data fra scraperen er ikke nødvendigvis et uafhængigt, repræsentativt tilfældigt sample.

In [12]:
import math

def wilson_demo(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k/n
    denom = 1 + z*z/n
    centre = (p + z*z/(2*n))/denom
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/denom
    return max(0.0, centre-half), min(1.0, centre+half)

for k, n in [(4, 5), (40, 100), (400, 1000)]:
    low, high = wilson_demo(k, n)
    print(f"{k}/{n} = {k/n:.1%}  |  Wilson 95%: {low:.1%}–{high:.1%}")

4/5 = 80.0%  |  Wilson 95%: 37.6%–96.4%
40/100 = 40.0%  |  Wilson 95%: 30.9%–49.8%
400/1000 = 40.0%  |  Wilson 95%: 37.0%–43.1%


**Output:** Den samme observerede andel på 40 % får typisk et smallere interval ved `n=1000` end ved `n=100`. Det viser, hvorfor appen også oplyser antal jobopslag.

**Alternativer:** Wald-interval, Clopper–Pearson, Jeffreys-interval, bootstrap. Wilson er et godt kompromis mellem enkel beregning og bedre dækning nær yderpunkter. Ingen af metoderne løser i sig selv scraperens udvalgsskævhed.

## 21. Hvordan finder appen demand for den valgte rolle?

I `demand.py` bruges `filter_postings(pt, role, levels=None, country=None)`:
- Først vælges `pt["role"] == role`.
- Hvis der er valgt `levels`, begrænses til de standardiserede niveauer.
- Eventuelt `country` anvendes case-insensitivt.
- `demand_table()` kalder derefter `demand_from_sets(sub["tool_set"])`.

I appen svarer `DATASETS["all"]` til primary-data på tværs af deres tilgængelige niveauer, mens `DATASETS["entry"]` bruger secondary med `["entry", "internship"]`.

**Vigtigt:** »Entry level« er ikke bare et filter oven på præcis det samme datasæt. Det skifter også til en **anden datakilde** med et andet scrape, andet landefokus og andet datagrundlag. Resultater på tværs af de to visninger bør derfor ikke fortolkes som en ren senioritetsforskel.

In [13]:
p = pd.DataFrame([
    {"role": "data scientist", "level": "entry", "tool_set": {"Python", "SQL"}},
    {"role": "data scientist", "level": "mid-senior", "tool_set": {"Python", "Docker"}},
    {"role": "data scientist", "level": "internship", "tool_set": {"SQL"}},
    {"role": "software engineer", "level": "entry", "tool_set": {"Java"}},
])
sel = p[(p["role"] == "data scientist") & (p["level"].isin(["entry", "internship"]))]
print("Antal valgte opslag:", len(sel))
print("Valgte tool_sets:", sel["tool_set"].tolist())

Antal valgte opslag: 2
Valgte tool_sets: [{'SQL', 'Python'}, {'SQL'}]


## 22. Hvordan hænger Del 2 sammen med Del 1 (appen)?

I appens `step_report()` sker i grove træk:

```python
demand = demand_table(get_postings(source), role, levels=levels)
rep = build_report(cv_tools, demand, top=15, min_share=0.05, note=note)
```

`get_postings(source)` bruger de forberedte CSV-filer, der caches, så appen ikke skal scanne de store jobbeskrivelser hver gang.

`build_report` er **næste logiske lag**: den deler topværktøjerne i `have` og `missing` og beregner coverage. Det er gennemgået i Del 1 og uddybes matematisk i Del 4.

Det er vigtigt at adskille:
- **Preprocessing (Del 2):** Hvilke opslag og værktøjer indgår?
- **Skill extraction/matching (Del 3):** Hvordan findes værktøjerne i tekst?
- **Ranking/statistik (Del 4):** Hvad betyder andele, intervaller og sammenligninger?

## 23. Samlet end-to-end eksempel uden Kaggle

Dette er et **syntetisk tracer bullet**, som følger arkitekturens rækkefølge:
`rå opslag → rolle → værktøjssæt → dubletfilter → demand`.

Eksemplet er bevidst lille og bruger en simple matchingordliste. Det er en **undervisningsmodel**, ikke en præcis genimplementering af `KeywordMatcher`.

In [14]:
example_rows = pd.DataFrame([
    {"id": 1, "title": "Data Scientist", "description": "Python and SQL required", "level": "Entry level"},
    {"id": 2, "title": "Data Scientist", "description": "Python and SQL required", "level": "Entry level"},
    {"id": 3, "title": "Data Scientist", "description": "Python and Docker", "level": "Mid-Senior level"},
    {"id": 4, "title": "Software Engineer", "description": "Java and Docker", "level": "Entry level"},
])
example_rows["role"] = pd.Series([np.nan]*len(example_rows), dtype="object")
for role_name, rx in patterns.items():
    hit = example_rows["role"].isna() & example_rows["title"].str.contains(rx, case=False, regex=True)
    example_rows.loc[hit, "role"] = role_name

example_rows["text_hash"] = example_rows["description"].map(text_hash_demo)
techs = ["Python", "SQL", "Docker", "Java"]
example_rows["tool_set"] = example_rows["description"].map(
    lambda s: {t for t in techs if re.search(r"\b" + re.escape(t) + r"\b", s, re.I)}
)
is_dup = example_rows.duplicated(["role", "text_hash"]) & example_rows["text_hash"].ne("")
processed = example_rows.loc[~is_dup].copy()
scientists = processed.loc[processed["role"] == "data scientist"]
n = len(scientists)
result = Counter(tool for tool_set in scientists["tool_set"] for tool in tool_set)
display(processed[["id", "role", "level", "tool_set"]])
display(pd.DataFrame([
    {"tool": tool, "n_with_tool": count, "share": count/n, "ci_95": wilson_demo(count, n)}
    for tool, count in result.items()
]).sort_values("share", ascending=False))

,id,role,level,tool_set
0,1,data scientist,Entry level,"{SQL, Python}"
2,3,data scientist,Mid-Senior level,"{Docker, Python}"
3,4,software engineer,Entry level,"{Docker, Java}"


,tool,n_with_tool,share,ci_95
1,Python,2,1.0,"(0.34237195288961925, 1.0)"
0,SQL,1,0.5,"(0.09452865480086614, 0.9054713451991339)"
2,Docker,1,0.5,"(0.09452865480086614, 0.9054713451991339)"


### Trinvis fortolkning af tracer bullet

1. To Data Scientist-rækker har samme beskrivelse og samme rolle, så den anden fjernes som genopslag.
2. Software Engineer indgår i den behandlede tabel, men ikke i den aktuelle Data Scientist-query.
3. Blandt de **to unikke** Data Scientist-opslag står Python i begge (100 %), SQL i ét (50 %) og Docker i ét (50 %).
4. Wilson-intervallerne bliver meget brede, fordi `n=2` er ekstremt lille.
5. En rapport bygget på kun to opslag ville derfor være meget usikker. Jeres app har også en advarsel, når der er under 100 opslag.

**Hvorfor er det et godt eksamenseksempel?** Du kan pege på præcis hvilke rækker, der tælles, hvilken der fjernes, og hvor hvert procenttal kommer fra.

## 24. Hvad fandt I konkret ud af i de to inspection notebooks?

### Primary-notebooken
- Kildens egen `job_skills.csv` indeholder **værktøjsnære labels**, men med mange varierende skrivemåder.
- Den store `job_summary.csv` bruges for opslagets egentlige fritekst; I behandler den i chunks.
- I sammenligner matcherens ekstraktion fra fritekst med de eksisterende label-lister.
- I inspicerer tidsdækning og januar/april-sammenligning, men ændrer siden scope væk fra trendpåstande.

### Secondary-notebooken
- Kildens »skills« er i høj grad **brede kategorier**, ikke specifikke værktøjer.
- I undersøger O*NET occupations, alternative job titles og software-skills.
- I tester en første keyword-baseline på et udvalg af posting-tekster.
- I undersøger, om de små sampleudsnit, rollenavne og manglende felter kan understøtte en prototype.

**Bemærk om kildeforskelle:** Inspection notebooks afspejler tidlige undersøgelser (uger 1–4). Den aktuelle pipeline i `demand.py` har senere fået eksplicit deduplikering og rettede rollemønstre. Til eksamen bør I forklare *udviklingen*, ikke påstå at den første notebook allerede indeholdt alle senere forbedringer.

## 25. Hvorfor I valgte netop denne arkitektur – og alternativerne

| Beslutning | Jeres valg | Hvorfor det passer | Alternativ og tradeoff |
|---|---|---|---|
| Datakilde | To eksisterende Kaggle-snapshots | Hurtig adgang til stor historisk jobtekst | Live scraping/API: nyere, men dyrere og mere skrøbeligt |
| Ordforråd | O*NET navngivne teknologier | Fælles, forklarbar repræsentation | Open vocabulary/ESCO: bredere, men anden mapping/evaluering |
| Rollemapping | Regex på titel | Transparent, let at reproducere | Embeddings/klassifikator: større fleksibilitet, mindre forklarbar |
| Store tekster | Chunked CSV-læsning | Reducerer memory-pres | DuckDB/Dask/Parquet: mere skalerbart, mere setup |
| Dubletter | SHA-1 af normaliseret tekst | Hurtig genkendelse af identiske annoncer | MinHash: finder near-duplicates, kan slå forskellige opslag sammen |
| Outputtabeller | To små CSV'er | Enkel deployment og hurtig app | SQL/Parquet: bedre ved vækst |
| Efterspørgsel | Andel af opslag med værktøj | Intuitiv, reproducerbar metric | TF-IDF/frekvens: mindre umiddelbart fortolkelig |
| Usikkerhed | Wilson 95 %-intervaller | Bedre for små samples / yderpunkter | Bootstrap/hierarkisk model: mere fleksibelt, mere komplekst |

## 26. Hvad er særligt vigtigt at kunne forsvare?

**1. »Hvorfor ikke træne en model til at estimere skill demand?«**  
I forsøger ikke at forudsige, hvor mange jobs der kommer i fremtiden. I beregner først et observeret deskriptivt mål: *andel eksisterende opslag, der nævner værktøjet*. En model er ikke nødvendig for selve denne optælling.

**2. »Hvorfor tælle én gang pr. posting?«**  
For at lange beskrivelser og gentagelser ikke giver ekstra vægt.

**3. »Hvorfor ikke bare bruge Kaggle-skill-kolonnen?«**  
I har to datasæt med inkompatible skill-labels. Især secondary har brede skill-kategorier, ikke specifikke værktøjer. Jeres egen fælles extraction giver sammenlignelighed mellem datasættene og CV.

**4. »Hvad gør O*NET?«**  
Det er et katalog/normaliseringsmål, ikke et estimat af jeres LinkedIn-data. Den faktiske efterspørgsel tælles fra jobteksterne.

**5. »Hvorfor lave en offline build?«**  
Appen kan indlæse små, behandlede filer i stedet for at analysere en flergigabyte-tekstfil ved hver brugerforespørgsel.

**6. »Er output repræsentativt for det danske jobmarked?«**  
Nej. Det er indsamlede LinkedIn-snapshots fra 2024, hovedsageligt USA. Det er en væsentlig generaliseringsbegrænsning.

**7. »Hvorfor ingen trendgraf?«**  
To snapshots fra forskellige scrapers er ikke tilstrækkeligt grundlag for at udlede en pålidelig tidsudvikling.

## 27. Øvespørgsmål til mundtlig eksamen

Prøv at besvare spørgsmålene **uden** at kigge på svarene ovenfor:

1. Hvad er forskellen på jeres `primary` og `secondary` datasæt?
2. Hvorfor kan I ikke kalde forskellen mellem januar og april for en trend?
3. Hvilke O*NET-filer har I undersøgt, og hvilken bruger appen reelt til matching?
4. Hvad er forskellen på `Hot Technology`-flaget og `share`?
5. Hvordan klassificerer `tag_role()` jobtitler, og hvad var UX-buggen?
6. Hvorfor normaliserer I experience levels?
7. Hvorfor læser I `job_summary.csv` i chunks?
8. Hvad er forskellen på at fjerne dubletter via `job_id` og via `text_hash`?
9. Hvad sker der, hvis en annonce skriver »Python« 10 gange?
10. Hvad betyder SQL-demand på 40 %?
11. Hvilke antagelser ligger bag et Wilson-konfidensinterval?
12. Hvorfor er en offline preprocessing-pipeline vigtig for appen?
13. Hvordan ville I gøre systemet mere relevant for danske jobs?
14. Hvilken slags fejl kunne et bredere ordforråd skabe?
15. Hvad er forskellen på en inspection-notebook og produktionskoden?

**Øvelse:** Udvælg et enkelt syntetisk opslag og gå hele vejen gennem dets transformationer: råtekst → rolle → matcher → kanoniske værktøjer → hash → gemt tabel → demand.

## 28. Et svar på 90 sekunder til eksamen

> »Vi startede med at undersøge to store LinkedIn-datasæt fra Kaggle. Det ene indeholder cirka 1,3 millioner opslag fra januar 2024 og giver relativt mange opslag pr. rolle. Det andet har omkring 124.000 opslag fra april 2024 og har blandt andet information om entry-level og internships. Vi fandt ud af, at data ikke var en pålidelig månedlig tidsserie, så vi opgav at påstå, at vi kunne måle skills, som trender op eller ned.
>
> Vi valgte O*NET som et fælles katalog over navngivne teknologier. Fordi O*NET-navnene ofte er lange, byggede vi en matcher med aliaser, der kan genkende fx SQL eller Power BI i jobbeskrivelser. Vi filtrerer jobopslag til fem definerede målroller ud fra regex på jobtitler.
>
> For hver relevant posting finder vi teknologierne og gemmer en lille behandlet tabel. Primary-teksten behandles i chunks, fordi den er meget stor. Vi bruger også en hash af normaliseret tekst til at fjerne gentagne annoncer, så de ikke tælles flere gange.
>
> Når appen kører, læser den disse færdige tabeller og beregner demand som antal relevante opslag med et værktøj divideret med alle relevante opslag. Vi viser også Wilson-konfidensintervaller. Derfor er den tunge databehandling adskilt fra den hurtige webapp. Resultaterne beskriver vores sample fra 2024 og ikke hele det danske arbejdsmarked.«

## 29. Kilder og præcis afgrænsning

**Projektkilder**
- [SkillRadar repository](https://github.com/Lucialululu/SkillRadar)
- [README.md](https://github.com/Lucialululu/SkillRadar/blob/main/README.md)
- [`skillradar/demand.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/demand.py)
- [`skillradar/matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/matcher.py)
- [`main.py`](https://github.com/Lucialululu/SkillRadar/blob/main/main.py)
- [`preprocessing/data_inspection_primary.ipynb`](https://github.com/Lucialululu/SkillRadar/blob/main/preprocessing/data_inspection_primary.ipynb)
- [`preprocessing/data_inspection_secondary.ipynb`](https://github.com/Lucialululu/SkillRadar/blob/main/preprocessing/data_inspection_secondary.ipynb)

**Eksterne baggrundskilder**
- [O*NET Database](https://www.onetcenter.org/database.html)
- [Primary Kaggle-datasæt](https://www.kaggle.com/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024)
- [Secondary Kaggle-datasæt](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings)

**Afgrænsning:** De små kørsler i denne notebook er selvstændige syntetiske demonstrationer, ikke genberegninger af jeres faktiske millionopslag. Resultattal fra README er angivet som projektets dokumenterede tal. Til detaljer om, hvordan `KeywordMatcher` er evalueret, og hvordan bootstrap/Spearman fungerer, fortsætter vi i Del 3 og Del 4.